## IE Results Viewer
Viewer for Information Extraction results on OASIS journal reports. Facilitates interactive adjustment of section and significance scores to affect results ranking.

In [ ]:
import io
import os
from ATRIUM_T4_1_2_IE_results_utils import *
from ipywidgets import Button, Output, FloatSlider, Layout, HBox, VBox, FileUpload
from IPython.display import display, HTML

# UI component for input data file selection
input_file_upload = FileUpload(
    button_style='primary',
    description="Select input file",  # Button text
    accept='.json',  # Accepted file extension
    multiple=False,  # True to accept multiple files upload else False
    layout=Layout(width='200px')
)

# when a file is selected display the selected file name 
file_name_display = Output()
def on_upload_change(change):
    file_name_display.clear_output()
    outputs.clear_output()    
    if input_file_upload.value:
        uploaded_file = input_file_upload.value[0]
        s = f"Selected: {uploaded_file.get('name', '-')}"
        with file_name_display:
            display(HTML(s))    
input_file_upload.observe(on_upload_change, names='value')

# UI controls - sliders for adjusting relevance scoring parameters
outputs2 = Output()
s_style = {'description_width': '150px', 'handle_color': 'lightblue'}
s_layout=Layout(width='500px')
slider_t = FloatSlider(description='Title score', value=DEFAULT_SCORES.get("title", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Score for terms appearing in the title section")
slider_a = FloatSlider(description='Abstract score', value=DEFAULT_SCORES.get("abstract", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Score for terms appearing in the abstract section")
slider_b = FloatSlider(description='Body score', value=DEFAULT_SCORES.get("body", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Score for terms appearing in the body section")
slider_s = FloatSlider(description='Significance score', value=DEFAULT_SCORES.get("sig_proximity", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Score for terms close to significance terms")
slider_mrs = FloatSlider(description='Minimum score', value=DEFAULT_SCORES.get("mrs", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Minimum relevance score for a term to be included in the results")
slider_mrc = FloatSlider(description='Minimum count', value=DEFAULT_SCORES.get("mrc", 1), min=1, max=50, step=1, layout=s_layout, style=s_style, tooltip="Minimum relevance count for a term to be included in the results")
sliders = VBox([slider_t, slider_a, slider_b, slider_s, slider_mrs, slider_mrc], layout=Layout(display='flex', flex_flow='column', gap='5px'))
# UI controls - buttons
b_layout = Layout(width='200px')
refresh = Button(description="Refresh results", icon='refresh', button_style='primary', layout=b_layout)
reset = Button(description="Clear and reset", icon='times', button_style='primary', layout=b_layout)
buttons = HBox([refresh, reset], layout=Layout(display='flex', flex_flow='row', gap='10px'))
# UI output for results table
outputs = Output()

# display all UI components
display(HBox([outputs2, input_file_upload, file_name_display]))
display(VBox([sliders, buttons, outputs]))

# refresh results when refresh button is clicked
def on_refresh_click(b):
    outputs.clear_output()
    input_file = input_file_upload.value[0] if input_file_upload.value else None
    if input_file:
        df = pd.DataFrame()        
        input_file_type = input_file.get('type','').strip().lower() 
        input_file_ext =  os.path.splitext(input_file.get('name','').strip().lower())             
        if input_file_type == "application/json" or input_file_ext == "json":  
            #data = json.load(io.StringIO(input_file['content'])) 
            data = json.load(io.BytesIO(input_file['content'])) 
            df = pd.DataFrame(data.get('spans', []))            
        
        # set any NaN values to blank string      
        #df.fillna("", inplace=True)  
        input_data = df.to_dict(orient='records')   
                        
        df = aggregate_results_by_concept(input_data, slider_t, slider_a, slider_b, slider_s) # aggregated results by concept id
        df = df[df['score'] >= slider_mrs.value]  # filtering by minimum relevance score
        df = df[df['count'] >= slider_mrc.value]  # filtering by minimum relevance count
        df = df.sort_values(by='score', ascending=False) #.head(20)
        df = df[['span', 'label', 'count', 'sec_score', 'sig_score', 'score']]
        styled_df = (df.style
            .set_caption("Results aggregated by concept")
            #.hide(subset=['id', 'text'], axis=1)
            .hide(axis="index")
            .highlight_max(subset=['sig_score', 'count']) #, color='red'
            .background_gradient(subset=['score', 'sec_score', 'sig_score', 'count']) #, cmap='YlOrRd', low=0.2, high=0.8
             #.applymap(color_negative_red)
            .format(na_rep="n/a")
            .format( "{:.2f}", subset=['sec_score', 'sig_score', 'score']))
       
        with outputs:
            display(HTML(styled_df.to_html(index=False)))   
refresh.on_click(on_refresh_click)

# clear fields and reset sliders when reset button is clicked
def on_reset_click(b):
    slider_t.value = DEFAULT_SCORES["title"]
    slider_a.value = DEFAULT_SCORES["abstract"]
    slider_b.value = DEFAULT_SCORES["body"]
    slider_s.value = DEFAULT_SCORES["sig_proximity"]
    slider_mrs.value = DEFAULT_SCORES["mrs"]
    slider_mrc.value = DEFAULT_SCORES["mrc"]
    outputs.clear_output()
    input_file_upload.value = []
    file_name_display.clear_output()
reset.on_click(on_reset_click)